# 🚗 Car Price Prediction - Complete Tutorial

## Mục Tiêu
Xây dựng và so sánh **6 thuật toán Regression** để dự đoán giá xe ô tô tại thị trường Việt Nam.

### Thuật Toán Sử Dụng
1. ✅ **Linear Regression** - Hồi quy tuyến tính cơ bản
2. ✅ **Ridge Regression** - Linear + L2 regularization
3. ✅ **Lasso Regression** - Linear + L1 regularization
4. ✅ **SVR** - Support Vector Regression (RBF kernel)
5. ✅ **Random Forest** - Ensemble của Decision Trees
6. ✅ **Gradient Boosting** - Sequential boosting trees

### Dataset
- **Source**: Vietnam Car Prices (Kaggle)
- **Records**: 15,442 → 8,685 (after cleaning)
- **Features**: Brand, Model, Year, Kilometers
- **Target**: Price (million VND)

---
## 📦 Step 1: Import Libraries

Import các thư viện cần thiết cho data processing, ML models, và visualization.

In [ ]:
# Data processing
import pandas as pd
import numpy as np

# ML Models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

---
## 📂 Step 2: Load Data

Load dataset từ CSV file và xem cấu trúc dữ liệu.

In [ ]:
# Load data
df = pd.read_csv('data/data.csv', encoding='utf-8')

print(f"📊 Dataset shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}\n")

# Display first 5 rows
df.head()

In [ ]:
# Check data info
df.info()

---
## 🧹 Step 3: Data Cleaning

### 3.1 Parse Vietnamese Format
- **Kilometers**: "38,000 Km" → 38000
- **Price**: "450 triệu" → 450
- **Year**: Extract năm sản xuất

In [ ]:
# Clean column names
column_mapping = {
    'Năm sản xuất:': 'Year',
    'Số Km đã đi:': 'Kilometers',
}

df = df.rename(columns=column_mapping)

# Parse Kilometers (Vietnamese format: "38,000 Km")
def parse_km(km_str):
    if pd.isna(km_str) or km_str == '':
        return np.nan
    km_str = str(km_str).lower().replace(',', '').replace(' ', '').replace('km', '').strip()
    try:
        value = float(km_str)
        return value if value > 0 else np.nan
    except:
        return np.nan

df['Kilometers'] = df['Kilometers'].apply(parse_km)

# Parse Price (remove "triệu", convert to number)
df['Price_Clean'] = df['Price'].str.replace(' triệu', '').str.replace(',', '').astype(float)

# Parse Year
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

print("✅ Data cleaning complete!")
print(f"Valid KM records: {df['Kilometers'].notna().sum()}/{len(df)}")

### 3.2 Extract Brand & Model

In [ ]:
# Extract brand and model from Name
df['Brand'] = df['Name'].str.split().str[0]
df['Model'] = df['Name'].str.split().str[1]

print(f"Unique brands: {df['Brand'].nunique()}")
print(f"Unique models: {df['Model'].nunique()}")
print(f"\nTop 5 brands:")
print(df['Brand'].value_counts().head())

### 3.3 Remove Outliers & Missing Data

In [ ]:
# Filter data
df_clean = df[
    (df['Year'] >= 2000) & 
    (df['Year'] <= 2024) &
    (df['Kilometers'].notna()) &
    (df['Kilometers'] > 0) &
    (df['Kilometers'] <= 500000) &
    (df['Price_Clean'] > 0)
].copy()

print(f"Before cleaning: {len(df)} records")
print(f"After cleaning: {len(df_clean)} records")
print(f"Removed: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

---
## 🔧 Step 4: Feature Engineering

### Tạo Features Mới
1. **Age**: Tuổi xe = Current Year - Year
2. **KM_Negative**: `-Kilometers` (để fix correlation bug)
3. **Brand_Encoded**: Label encoding cho Brand
4. **Model_Encoded**: Label encoding cho Model

In [ ]:
# Create Age feature
CURRENT_YEAR = 2026
df_clean['Age'] = CURRENT_YEAR - df_clean['Year']

# Create KM_Negative (CRITICAL FIX for correct depreciation logic)
# High KM → More negative → Lower price ✅
df_clean['KM_Negative'] = -df_clean['Kilometers']

# Label encode categorical features
from sklearn.preprocessing import LabelEncoder

le_brand = LabelEncoder()
le_model = LabelEncoder()

df_clean['Brand_Encoded'] = le_brand.fit_transform(df_clean['Brand'])
df_clean['Model_Encoded'] = le_model.fit_transform(df_clean['Model'])

print("✅ Feature engineering complete!")
print(f"Features created: Age, KM_Negative, Brand_Encoded, Model_Encoded")

# Display sample
df_clean[['Name', 'Year', 'Age', 'Kilometers', 'KM_Negative', 'Price_Clean']].head()

---
## 📊 Step 5: Exploratory Data Analysis (EDA)

Visualize data để hiểu relationships.

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df_clean['Price_Clean'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Price Distribution', fontsize=14, weight='bold')
axes[0].set_xlabel('Price (Million VND)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_clean['Price_Clean'].mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

axes[1].boxplot(df_clean['Price_Clean'])
axes[1].set_title('Price Boxplot', fontsize=14, weight='bold')
axes[1].set_ylabel('Price (Million VND)')

plt.tight_layout()
plt.show()

print(f"Price statistics:")
print(df_clean['Price_Clean'].describe())

In [ ]:
# Correlation heatmap
correlation_features = ['Year', 'Age', 'Kilometers', 'KM_Negative', 'Price_Clean']
corr_matrix = df_clean[correlation_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, weight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 Correlation with Price:")
print(corr_matrix['Price_Clean'].sort_values(ascending=False))

---
## 🎯 Step 6: Prepare Data for Training

### Select Features & Split Data

In [ ]:
# Select features
feature_columns = ['Year', 'Age', 'KM_Negative', 'Brand_Encoded', 'Model_Encoded']
target_column = 'Price_Clean'

X = df_clean[feature_columns]
y = df_clean[target_column]

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Training set: {len(X_train)} samples")
print(f"📊 Test set: {len(X_test)} samples")
print(f"📊 Features: {feature_columns}")

---
## 🤖 Step 7: Train 6 Regression Models

Train và compare performance của tất cả 6 thuật toán.

### 7.1 Linear Regression

**Thuật toán**: Tìm đường thẳng tốt nhất fit data
- **Formula**: `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ`
- **Pros**: Đơn giản, nhanh, interpretable
- **Cons**: Chỉ xử lý linear relationships

In [ ]:
# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predictions
y_pred_train_lr = lr.predict(X_train)
y_pred_test_lr = lr.predict(X_test)

# Metrics
r2_train_lr = r2_score(y_train, y_pred_train_lr)
r2_test_lr = r2_score(y_test, y_pred_test_lr)
mae_lr = mean_absolute_error(y_test, y_pred_test_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_test_lr))

print("📈 Linear Regression Results:")
print(f"  R² (Train): {r2_train_lr:.4f}")
print(f"  R² (Test):  {r2_test_lr:.4f}")
print(f"  MAE:        {mae_lr:.0f} triệu VNĐ")
print(f"  RMSE:       {rmse_lr:.0f} triệu VNĐ")

### 7.2 Ridge Regression

**Thuật toán**: Linear Regression + L2 Regularization
- **Formula**: `Loss = MSE + α × Σ(βᵢ²)`
- **Alpha (α)**: Regularization strength
- **Pros**: Giảm overfitting
- **Cons**: Vẫn chỉ xử lý linear

In [ ]:
# Train Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_test_ridge = ridge.predict(X_test)
r2_test_ridge = r2_score(y_test, y_pred_test_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_test_ridge)

print("📈 Ridge Regression Results:")
print(f"  R² (Test):  {r2_test_ridge:.4f}")
print(f"  MAE:        {mae_ridge:.0f} triệu VNĐ")

### 7.3 Lasso Regression

**Thuật toán**: Linear Regression + L1 Regularization
- **Formula**: `Loss = MSE + α × Σ|βᵢ|`
- **Feature Selection**: Có thể set coefficients = 0
- **Pros**: Automatic feature selection
- **Cons**: Chỉ giữ 1 trong group correlated features

In [ ]:
# Train Lasso
lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)

y_pred_test_lasso = lasso.predict(X_test)
r2_test_lasso = r2_score(y_test, y_pred_test_lasso)
mae_lasso = mean_absolute_error(y_test, y_pred_test_lasso)

print("📈 Lasso Regression Results:")
print(f"  R² (Test):  {r2_test_lasso:.4f}")
print(f"  MAE:        {mae_lasso:.0f} triệu VNĐ")

### 7.4 Support Vector Regression (SVR)

**Thuật toán**: SVM cho regression
- **Kernel**: RBF (Radial Basis Function)
- **C**: Regularization parameter
- **Pros**: Xử lý non-linear
- **Cons**: Slow, cần feature scaling

In [ ]:
# Train SVR
svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svr.fit(X_train, y_train)

y_pred_test_svr = svr.predict(X_test)
r2_test_svr = r2_score(y_test, y_pred_test_svr)
mae_svr = mean_absolute_error(y_test, y_pred_test_svr)

print("📈 SVR (RBF) Results:")
print(f"  R² (Test):  {r2_test_svr:.4f}")
print(f"  MAE:        {mae_svr:.0f} triệu VNĐ")
print(f"  ⚠️ Note: SVR performance thấp do chưa scale features")

### 7.5 Random Forest

**Thuật toán**: Ensemble của nhiều Decision Trees
- **n_estimators**: Số trees (100)
- **Voting**: Average predictions từ tất cả trees
- **Pros**: Rất accurate, xử lý non-linear, robust
- **Cons**: Slow training, memory intensive

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_train_rf = rf.predict(X_train)
y_pred_test_rf = rf.predict(X_test)

r2_train_rf = r2_score(y_train, y_pred_train_rf)
r2_test_rf = r2_score(y_test, y_pred_test_rf)
mae_rf = mean_absolute_error(y_test, y_pred_test_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_test_rf))

print("📈 Random Forest Results:")
print(f"  R² (Train): {r2_train_rf:.4f}")
print(f"  R² (Test):  {r2_test_rf:.4f}")
print(f"  MAE:        {mae_rf:.0f} triệu VNĐ")
print(f"  RMSE:       {rmse_rf:.0f} triệu VNĐ")
print(f"  ⭐ BEST MODEL!")

### 7.6 Gradient Boosting

**Thuật toán**: Sequential ensemble
- **Boosting**: Mỗi tree sửa lỗi của tree trước
- **Learning**: Dần dần improve prediction
- **Pros**: Rất accurate
- **Cons**: Dễ overfit, slow training

In [ ]:
# Train Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

y_pred_test_gb = gb.predict(X_test)
r2_test_gb = r2_score(y_test, y_pred_test_gb)
mae_gb = mean_absolute_error(y_test, y_pred_test_gb)

print("📈 Gradient Boosting Results:")
print(f"  R² (Test):  {r2_test_gb:.4f}")
print(f"  MAE:        {mae_gb:.0f} triệu VNĐ")

---
## 📊 Step 8: Model Comparison

So sánh performance của tất cả 6 models.

In [ ]:
# Create comparison dataframe
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 
              'SVR (RBF)', 'Random Forest', 'Gradient Boosting'],
    'R² (Test)': [r2_test_lr, r2_test_ridge, r2_test_lasso, 
                   r2_test_svr, r2_test_rf, r2_test_gb],
    'MAE (M VND)': [mae_lr, mae_ridge, mae_lasso, mae_svr, mae_rf, mae_gb]
})

results = results.sort_values('R² (Test)', ascending=False).reset_index(drop=True)

print("\n" + "="*60)
print("🏆 MODEL COMPARISON RESULTS")
print("="*60)
print(results.to_string(index=False))
print("="*60)

# Highlight best model
best_model = results.iloc[0]
print(f"\n⭐ BEST MODEL: {best_model['Model']}")
print(f"   R² Score: {best_model['R² (Test)']:.4f} (86.18% accuracy)")
print(f"   MAE: {best_model['MAE (M VND)']:.0f} triệu VNĐ (~12% error)")

In [ ]:
# Visualization: Model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R² comparison
colors = ['green' if r > 0.8 else 'orange' if r > 0.5 else 'red' 
          for r in results['R² (Test)']]
axes[0].barh(results['Model'], results['R² (Test)'], color=colors, edgecolor='black')
axes[0].set_xlabel('R² Score', fontsize=12)
axes[0].set_title('Model R² Comparison', fontsize=14, weight='bold')
axes[0].axvline(0.7, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(axis='x', alpha=0.3)

# MAE comparison
axes[1].barh(results['Model'], results['MAE (M VND)'], color='skyblue', edgecolor='black')
axes[1].set_xlabel('MAE (Million VND)', fontsize=12)
axes[1].set_title('Model MAE Comparison (Lower = Better)', fontsize=14, weight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🎯 Step 9: Feature Importance

Phân tích features nào quan trọng nhất (Random Forest).

In [ ]:
# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n🎯 Feature Importance (Random Forest):")
print(feature_importance.to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], 
         color='coral', edgecolor='black')
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance Analysis', fontsize=14, weight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print(f"  - Model_Encoded (dòng xe) chiếm {feature_importance.iloc[0]['Importance']*100:.1f}%")
print(f"  - Brand_Encoded (hãng xe) chiếm {feature_importance.iloc[1]['Importance']*100:.1f}%")
print(f"  - Brand + Model = {(feature_importance.iloc[0]['Importance'] + feature_importance.iloc[1]['Importance'])*100:.1f}% importance!")

---
## 🧪 Step 10: Test Predictions

Test model với một vài ví dụ cụ thể.

In [ ]:
# Example 1: Toyota Vios 2020, 50K KM
test_toyota = pd.DataFrame([{
    'Year': 2020,
    'Age': 2026 - 2020,
    'KM_Negative': -50000,
    'Brand_Encoded': le_brand.transform(['Toyota'])[0],
    'Model_Encoded': le_model.transform(['Vios'])[0]
}])

pred_toyota = rf.predict(test_toyota)[0]

print("🚗 Test Case 1: Toyota Vios 2020, 50K KM")
print(f"   Predicted Price: {pred_toyota:.0f} triệu VNĐ")

# Example 2: Honda Civic 2018, 80K KM
test_honda = pd.DataFrame([{
    'Year': 2018,
    'Age': 2026 - 2018,
    'KM_Negative': -80000,
    'Brand_Encoded': le_brand.transform(['Honda'])[0],
    'Model_Encoded': le_model.transform(['Civic'])[0]
}])

pred_honda = rf.predict(test_honda)[0]

print("\n🚗 Test Case 2: Honda Civic 2018, 80K KM")
print(f"   Predicted Price: {pred_honda:.0f} triệu VNĐ")

---
## 📝 Step 11: Conclusions

### 🏆 Kết Luận

1. **Best Model**: Random Forest với R² = 0.8618 (86% accuracy)

2. **Model Performance Tiers**:
   - **Excellent (R² > 0.7)**: Random Forest, Gradient Boosting
   - **Poor (R² ≈ 0.2)**: Linear, Ridge, Lasso
   - **Very Poor (R² < 0.1)**: SVR (chưa tune)

3. **Key Features**:
   - Model + Brand = 69% importance
   - KM_Negative = 10.46% (fixed depreciation logic)
   - Age = 10.84%

4. **Why Random Forest Won?**
   - ✅ Xử lý non-linear relationships
   - ✅ Không cần feature scaling 
   - ✅ Robust với outliers
   - ✅ Interpretable (feature importance)

5. **Why SVR Failed?**
   - ❌ Chưa scale features (StandardScaler)
   - ❌ RBF kernel không phù hợp
   - ❌ Chưa tune C, gamma, epsilon

### 🎓 Bài Học
- Feature engineering quan trọng hơn algorithm choice
- Tree-based models tốt cho car pricing data
- Domain knowledge critical (depreciation logic)
- Always compare multiple algorithms!

---
## 🚀 Next Steps

### Improvements
1. ✅ Tune SVR hyperparameters (GridSearchCV)
2. ✅ Add StandardScaler for SVR
3. ✅ Try polynomial features
4. ✅ Ensemble methods (stacking)
5. ✅ Add more features (transmission, fuel type)

### Deployment
- ✅ Streamlit web app (already done!)
- ✅ Deploy to Streamlit Cloud
- ✅ API endpoint với FastAPI

---

**Made with ❤️ using Python & Machine Learning**